In [1]:
pip install spacy 

Note: you may need to restart the kernel to use updated packages.


In [4]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.7 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import spacy 
nlp = spacy.load('en_core_web_sm') 
print('spaCy ready!')

spaCy ready!


****PART 1: REGULAR EXPRESSIONS****

****Task 1.1: Extract Dates****

In [8]:
import re 
from datetime import datetime

def extract_dates(text):
    """
    Extract dates in various formats
    Formats: MM/DD/YYYY, DD-MM-YYYY, Month DD, YYYY, YYYY-MM-DD
    """
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',  # MM/DD/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',  # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',  # Month DD, YYYY
        r'\d{4}-\d{2}-\d{2}'  # ISO format
    ]
    
    dates = []
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)
    
    return dates

# Test
text = "Invoice date: 03/15/2024. Due: March 30, 2024"
print(extract_dates(text))

['03/15/2024', 'March 30, 2024']


****Task 1.2: Extract Currency Amounts****

In [9]:
import re

def extract_amounts(text):
    """
    Extract currency amounts
    Handles: $1,250.50, 1250.50, $1250
    """
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    
    amounts = re.findall(pattern, text)
    
    # Convert to float
    cleaned = []
    for amount in amounts:
        # Remove $ and commas
        clean = amount.replace('$', '').replace(',', '')
        cleaned.append(float(clean))
    
    return cleaned

# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 1125.45"
print(extract_amounts(text))


[1250.5, 125.05, 112.0, 5.45]


****Task 1.3: Extract Invoice/Order Numbers****

In [10]:
import re

def extract_invoice_number(text):
    """
    Extract invoice/order numbers
    Patterns: INV-2024-001, #12345, ORDER-ABC123
    """
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Return full match or first captured group
            return match.group(1) if match.groups() else match.group(0)
    
    return None

# Test
text = "Invoice Number: INV-2024-001"
print(extract_invoice_number(text))


INV-2024-001


****PART 2: NAMED ENTITY RECOGNITION****

****Task 2.1: Basic NER with spaCy****

In [11]:
import spacy

# Load model
nlp = spacy.load('en_core_web_sm')

# Sample invoice text
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

# Process text
doc = nlp(text)

# Extract entities
print('Found entities:')
for ent in doc.ents:
    print(f'{ent.text:20} {ent.label_:15} {spacy.explain(ent.label_)}')

Found entities:
Acme Corporation     ORG             Companies, agencies, institutions, etc.
123                  CARDINAL        Numerals that do not fall under another type
Main Street          FAC             Buildings, airports, highways, bridges, etc.
New York             GPE             Countries, cities, states
10001                DATE            Absolute or relative dates or periods
John Smith           PERSON          People, including fictional
March 15, 2024       DATE            Absolute or relative dates or periods
1,250.50             MONEY           Monetary values, including unit


****Task 2.2: Extract Specific Entity Types****

In [12]:
def extract_entities(text):
    """
    Extract and organize entities by type
    """
    doc = nlp(text)
    
    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }
    
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)
    
    return entities
   
# Test
result = extract_entities(text)

for entity_type, values in result.items():
    print(f'{entity_type}: {values}')

persons: ['John Smith']
organizations: ['Acme Corporation']
locations: ['New York']
dates: ['10001', 'March 15, 2024']
money: ['1,250.50']


****Task 2.3: Visualize Entities with displaCy****

In [13]:
from spacy import displacy

html = displacy.render(doc, style='ent')

with open('entities.html', 'w', encoding='utf-8') as f:
    f.write(str(html))  # force string conversion

print('Visualization saved to entities.html')

Visualization saved to entities.html


****PART 3: COMPLETE EXTRACTION PIPELINE**** 

****Task 3.1: Build Invoice Processor****

In [19]:
import json
import pytesseract
from PIL import Image

def process_invoice(image_path):
    """
    Complete pipeline: OCR → Extraction → JSON
    """
    # Step 1: OCR
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)
    
    # Step 2: Extract with regex
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }
    
    # Step 3: Extract with NER
    entities = extract_entities(text)
    invoice_data.update(entities)
    
    # Step 4: Post-process
    if invoice_data['amounts']:
        invoice_data['total_amount'] = max(invoice_data['amounts'])
    
    if invoice_data['dates']:
        invoice_data['invoice_date'] = invoice_data['dates'][0]
         
    return invoice_data


# Test on sample
result = process_invoice('/kaggle/input/datasets/trainingdatapro/ocr-receipts-text-detection/images/11.jpg')
print(json.dumps(result, indent=2))
{
  "invoice_number": "INV-2024-001",
  "dates": [
    "10001",
    "555-1234",
    "98765432100124"
  ],
  "amounts": [
    123.0,
    100.0,
    1.0,
    212.0,
    555.0,
    123.0,
    4.0,
    202.0,
    4.0,
    1.0,
    24.0,
    15.0,
    3.0,
    202.0,
    4.0,
    11.0,
       23.0,
    135.95,
    8.88,
    12.08,
    148.03,
    567.0,
    8.0,
    987.0,
    654.0,
    321.0,
    987.0,
    654.0,
    321.0,
    1.0,
    24.0
  ],
  "persons": [
    "Wireless Mouse"
  ],
  "organizations": [],
  "locations": [
    "New York"
  ],
  "money": [
    "135.95",
    "12.08",
    "148.03"
  ],
  "total_amount": 987.0,
  "invoice_date": "10001"
}

{
  "invoice_number": null,
  "dates": [
    "4040",
    "156415",
    "01",
    "94135"
  ],
  "amounts": [
    6.0,
    3.0,
    1.0,
    160.0,
    2.0,
    2.0,
    0.0,
    3.0,
    8.0,
    2.0,
    2.0,
    2.0,
    1.0,
    1.08,
    1.0,
    1.99,
    2.15,
    404.0,
    0.0,
    3.99,
    3.99,
    156.0,
    415.0,
    7.0,
    1.0,
    1.0,
    326.0,
    941.0,
    35.0,
    93.0,
    45.44
  ],
  "persons": [],
  "organizations": [
    "BLACK CV 2.15",
    "NHP"
  ],
  "locations": [
    "PL TORTILLA",
    "California"
  ],
  "money": [
    "2"
  ],
  "total_amount": 941.0,
  "invoice_date": "4040"
}


{'invoice_number': 'INV-2024-001',
 'dates': ['10001', '555-1234', '98765432100124'],
 'amounts': [123.0,
  100.0,
  1.0,
  212.0,
  555.0,
  123.0,
  4.0,
  202.0,
  4.0,
  1.0,
  24.0,
  15.0,
  3.0,
  202.0,
  4.0,
  11.0,
  23.0,
  135.95,
  8.88,
  12.08,
  148.03,
  567.0,
  8.0,
  987.0,
  654.0,
  321.0,
  987.0,
  654.0,
  321.0,
  1.0,
  24.0],
 'persons': ['Wireless Mouse'],
 'organizations': [],
 'locations': ['New York'],
 'money': ['135.95', '12.08', '148.03'],
 'total_amount': 987.0,
 'invoice_date': '10001'}

****Task 3.2: Save Results as JSON****

In [20]:
# Save to JSON file
output_file = 'extracted_data(Receipt_1).json'

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2)

print(f'Results saved to {output_file}')

Results saved to extracted_data(Receipt_1).json


In [23]:
import json
import pytesseract
from PIL import Image

def process_invoice(image_path):
    """
    Complete pipeline: OCR → Extraction → JSON
    """
    # Step 1: OCR
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)
    
    # Step 2: Extract with regex
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }
    
    # Step 3: Extract with NER
    entities = extract_entities(text)
    invoice_data.update(entities)
    
    # Step 4: Post-process
    if invoice_data['amounts']:
        invoice_data['total_amount'] = max(invoice_data['amounts'])
    
    if invoice_data['dates']:
        invoice_data['invoice_date'] = invoice_data['dates'][0]
    
    return invoice_data
    
# Test on sample
result = process_invoice('/kaggle/input/datasets/trainingdatapro/ocr-receipts-text-detection/images/12.jpg')
print(json.dumps(result, indent=2))

{
  "invoice_number": "#007190",
  "dates": [
    "007190",
    "October 30th"
  ],
  "amounts": [
    636.0,
    536.0,
    6.0,
    6.0,
    260.0,
    0.0,
    0.0,
    18.0,
    55.0,
    65.0,
    356.0,
    920.0,
    144.0,
    6.63,
    8.0,
    17.0,
    4.0,
    1.0,
    3.0,
    0.0,
    0.54,
    3.0,
    4.35,
    0.0,
    0.76,
    18.75,
    18.75,
    354.0,
    6.0,
    7.0,
    190.0,
    0.0,
    3.0,
    30.0,
    10.0,
    20.0,
    7.0,
    13.0,
    48.0,
    43.0
  ],
  "persons": [
    "WAL*MART",
    "Bs X\nWOMEN"
  ],
  "organizations": [
    "IESTERFIELD",
    "Cee"
  ],
  "locations": [
    "MO 6"
  ],
  "money": [
    "2600",
    "###"
  ],
  "total_amount": 920.0,
  "invoice_date": "007190"
}
